# 데이터 분석 코드를 활용하는 챗봇

## 실습 목표
---
로컬에서 동작하는 오픈 소스 기반 LLM이 주어진 요청과 데이터에 맞춰 데이터를 분석하는 코드를 생성하고, 그 코드를 실행한 분석 결과를 바탕으로 답변하는 챗봇을 개발합니다.

## 실습 목차
---

1. **데이터 특징 추출:** 주어진 데이터가 어떻게 구성되어 있는지 추출하고 이를 시스템 프롬프트에 적용합니다.

2. **데이터 분석 체인 구성:** 데이터를 분석하는 코드를 생성 및 실행하고, 그 결과를 출력하는 체인을 구성합니다.

3. **데이터 그래프 생성 체인 구성:** 데이터를 분석하여 대응하는 그래프를 생성하는 코드를 생성하고 실행하는 체인을 구성합니다.

## 0. 환경 설정
- 필요한 라이브러리를 불러옵니다.

In [2]:
import contextlib
import io
import os

import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langchain_ollama import ChatOllama, OllamaEmbeddings

/var/folders/m9/22pbfsqj2k52h8fn3krgqw740000gn/T/ipykernel_90832/979442375.py:9: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.tools.python.tool import PythonAstREPLTool


- Ollama를 통해 llama 3.1 8B 모델을 불러옵니다.

In [3]:
!ollama pull llama3.1

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 


## 1. 데이터 특징 추출
- 주어진 데이터가 어떻게 구성되어 있는지 추출하고 이를 시스템 프롬프트에 적용합니다.
- 이번 실습에 사용할 데이터는 머신러닝 회귀 분석에 활용할 수 있는 잉크젯 데이터입니다.

먼저, llama 3.1 모델을 사용하는 ChatOllama 객체와 OllamaEmbeddings 객체를 생성합니다.

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings

In [5]:
# 현업에서 ollama를 통해 로컬 LLM을 활용하기 위해서는 ollama serve 명령어를 통해 ollama instance를 실행해야 합니다.
# 현재 실습 환경에서는 nohup ollama serve & 명령어를 통해 백그라운드에 ollama instance를 실행한 상태입니다. 
llm = ChatOllama(model="llama3.1")
#embeddings = OllamaEmbeddings(model="llama3.1")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8848.74it/s]


다음으로, 데이터를 불러오고, 데이터의 컬럼명을 변수에 저장합니다.

대부분의 경우 컬럼명에서 데이터의 특성을 파악할 수 있기 때문에, LLM이 사용자의 질문에 맞는 데이터를 어떤 코드를 통해 추출할지 추론하는 좋은 단서가 됩니다.

In [6]:
# 데이터를 불러오고, 이름과 컬럼명을 저장합니다.
data_dir = './data'
df_inkjet = pd.read_csv(os.path.join(data_dir, 'InkjetDB_preprocessing.csv'), index_col=0)

# 데이터를 저장한 변수명을 LLM에 제공하여 이 변수를 활용하는 코드를 작성하게 할 수 있습니다.
df_name = "df_inkjet"
df_columns = ", ".join(df_inkjet.columns)

다음으로, 데이터의 컬럼명을 바탕으로 코드를 생성하는 프롬프트를 작성합니다.

In [7]:
system_message = (
    "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n" # 역할(페르소나) 부여
    f"{df_name} DataFrame으로 주어진 데이터에서 질문에 답할 수 있는 정보를 출력하는 파이썬 코드를 작성하세요.\n" # 역할 설명
    f"{df_name} DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n" # 데이터 정보 제공
    "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다." # 추가 지시사항
)

message_with_data_info = [
    ("system", system_message),
    ("human", "{question}"),
]

시스템 프롬프트를 확인해 봅시다.

In [8]:
print(system_message)

당신은 주어진 데이터를 분석하는 데이터 분석가입니다.
df_inkjet DataFrame으로 주어진 데이터에서 질문에 답할 수 있는 정보를 출력하는 파이썬 코드를 작성하세요.
df_inkjet DataFrame에는 다음과 같은 열이 있습니다: Viscosity, Velocity, PrintingSpeed, PatternSize
데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다.


이전 실습에서 구성한 체인과 비슷하게, 코드를 생성하는 체인을 구성하고 실행해 봅시다.

In [9]:
prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

# 체인 구성
code_gen_chain = (
    {"question": RunnablePassthrough()}
    | prompt_with_data_info
    | llm
    | StrOutputParser()
)

print(code_gen_chain.invoke("Velocity가 가장 큰 데이터를 찾아줘"))

당신의 도움을 청길세라 다음과 같은 코드를 사용해 볼 수 있습니다.

```python
# Velocity가 가장 큰 데이터 찾기
max_velocity_data = df_inkjet.loc[df_inkjet['Velocity'].idxmax()]

print("Velocity가 가장 큰 데이터는:")
print(max_velocity_data)
```

이 코드는 `df_inkjet`의 'Velocity' 열에서 최대값을 찾아 해당 행의 데이터를 출력합니다.


그럴듯한 코드를 생성했습니다. 이제 이 코드를 복사해서 실행해보세요.
- 앞선 코드에서 `df_inkjet` 데이터를 불러왔기 때문에, 생성된 코드를 그대로 복사-붙여넣기 한 후 실행해주시면 됩니다. 또한 답변 중 코드 부분만 발췌해서 실행해야 합니다. 

In [10]:
# 체인의 답변 중 코드 부분만 추출해서 복사 붙여넣기 해주세요.
# Velocity가 가장 큰 데이터 찾기
max_velocity_data = df_inkjet.loc[df_inkjet['Velocity'].idxmax()]

print("Velocity가 가장 큰 데이터는:")
print(max_velocity_data)
# Velocity가 가장 큰 데이터 찾기
# max_velocity = df_inkjet['Velocity'].max()

# print(f"Velocity가 가장 큰 데이터: {max_velocity}")

Velocity가 가장 큰 데이터는:
Viscosity          8
Velocity           9
PrintingSpeed    250
PatternSize       14
Name: 125, dtype: int64


```bash
Viscosity          8
Velocity           9
PrintingSpeed    250
PatternSize       14
Name: 125, dtype: int64
```
위와 같은 텍스트가 출력되면 성공입니다. 만약 다른 텍스트가 출력된다면 코드를 유지한 채 2~3번 더 실행해보세요.

목적에 맞게 잘 데이터를 추출한 것을 확인할 수 있습니다. 

이제 이 코드를 수동으로 실행하지 않고, 체인을 통해 실행하고 그 결과만 바로 받아볼 수 있도록 데이터 분석 체인을 구성해봅시다.

## 2. 데이터 분석 체인 구성
- 데이터를 분석하는 코드를 생성하고, 코드를 실행하고, 그 결과를 답변하는 체인을 구성합니다.

조금 전 저희는 코드 생성 체인이 생성한 코드를 수동으로 복사하여 코드 블럭에 입력한 후 결과를 확인했습니다.

이 과정을 자동화 하여 자연어로 원하는 데이터를 입력하면, 그에 맞춘 데이터만 나오는 체인을 구성해 봅시다.

### 2-1. 코드 복사
먼저, LLM이 생성한 코드를 찾아서 복사하는 기능을 추가해 봅시다.

입력된 텍스트에서 파이썬 코드만 추출하는 함수를 정의합니다.

In [13]:
def python_code_parser(input: str) -> str:
    # LLM은 대부분 ``` 블럭 안에 코드를 출력합니다. 이를 활용합니다.
    # ```python (코드) ```, 혹은 ``` (코드) ``` 형태로 출력됩니다. 두 경우 모두에 대응하도록 코드를 작성합니다.
    processed_input = input.replace("```python", "```").strip()
    parsed_input_list = processed_input.split("```")

    # 만약 ``` 블럭이 없다면, 입력 텍스트 전체가 코드라고 간주합니다.
    # 아닐 경우 이어지는 코드 실행 과정에서 예외 처리를 통해 오류를 확인할 수 있습니다.
    if len(parsed_input_list) == 1:
        return processed_input

    # 코드 부분만 추출합니다. 
    # LLM은 여러 코드 블럭에 걸쳐 필요한 코드를 출력할 수 있으므로, 코드가 있는 홀수 번째 텍스트를 모두 저장합니다.
    parsed_code_list = []
    for i in range(1, len(parsed_input_list), 2):
        parsed_code_list.append(parsed_input_list[i])
    
    # 코드 부분을 하나로 합칩니다.
    return "\n".join(parsed_code_list)

이제 함수를 체인에 연결합니다.
- 단순 함수를 체인에 연결하면, LangChain은 암묵적으로 "Runnable"한 `RunnableLambda` 로 변환해서 체인에 연결합니다.

In [14]:
code_gen_chain_with_parser = (
    code_gen_chain
    | python_code_parser
)

아까랑 같은 질문을 해봅시다.

In [15]:
print(code_gen_chain_with_parser.invoke("Velocity가 가장 큰 데이터를 찾아줘"))


print(df_inkjet.loc[df_inkjet['Velocity'].idxmax()])


Viscosity     0.5
Velocity      10
PrintingSpeed  2
PatternSize   1
Name: index, dtype: object


max_velocity_rows = df_inkjet[df_inkjet['Velocity'] == df_inkjet['Velocity'].max()]
print(max_velocity_rows)



생성하는 코드는 비슷하지만, 앞뒤로 설명이 없고 코드 부분만 깔끔하게 출력될 것입니다. 만약 그렇지 않을 경우, 다시 한번 실행해주세요.

### 2-2. 코드 실행
- 코드만 깔끔하게 추출하는 기능을 구현했으니, 이제 이 코드를 실행해봅시다.

파이썬에는 코드 텍스트를 실행하는 `exec()` 함수가 있습니다. 이 함수를 활용하여 입력된 코드를 실행하고, 그 출력 결과를 반환하는 함수를 정의합니다.

In [16]:
def run_code(input_code: str):
    # 코드가 출력한 값을 캡쳐하기 위한 StringIO 객체를 생성합니다.
    output = io.StringIO()
    try:
        # stdout 출력을 StringIO Object로 리디렉션 하는 블럭을 구성합니다.
        with contextlib.redirect_stdout(output):
            # Python 3.10 버전에서는 exec 함수에 키워드 인자를 사용할 수 없습니다.
            # 코드가 실행하면서 출력한 모든 결과를 캡쳐합니다.
            exec(input_code, {"df_inkjet": df_inkjet})
    except Exception as e:
        # 에러가 발생할 경우, 이를 StringIO 객체에 저장합니다.
        print(f"Error: {e}", file=output)
    # StringIO 객체에 저장된 값을 반환합니다.
    return output.getvalue()

아까와 동일하게 함수를 체인에 연결합니다.

In [17]:
code_execute_chain = (
    code_gen_chain_with_parser |
    run_code
)

같은 질문을 한번 더 입력해 봅시다.

In [18]:
print(code_execute_chain.invoke("Velocity가 가장 큰 데이터를 찾아줘"))

Viscosity          8
Velocity           9
PrintingSpeed    250
PatternSize       14
Name: 125, dtype: int64



이번에는 코드 실행 결과만 반환하는 것을 확인할 수 있습니다.

만약 위와 같이 `stdout`을 캡쳐하는 것이 불가능하다면, Langchain-experimental의 `PythonAstREPLTool` 을 활용할 수도 있습니다.

단, `PythonAstREPLTool`은 코드가 가장 마지막에 출력한 출력 값만 반환합니다.

In [19]:
repl_execute_chain = (
    code_gen_chain_with_parser |
    PythonAstREPLTool(locals={"df_inkjet": df_inkjet})
)

In [20]:
print(repl_execute_chain.invoke("Velocity가 가장 큰 데이터를 찾아줘"))

Viscosity          8
Velocity           9
PrintingSpeed    250
PatternSize       14
Name: 125, dtype: int64



저희는 원하는 데이터를 자연어로 입력하면, 이를 바탕으로 자동으로 코드를 생성하고 결과를 반환하는 체인을 구성했습니다. 

그러나, 현재 구성한 체인은 **특정 데이터를 찾아달라는 질문 뿐만 아니라, 이를 설명해달라는 질문에도 데이터만 반환합니다.**

In [21]:
# exec() 함수를 활용하는 체인
print(code_execute_chain.invoke("Velocity가 가장 큰 데이터에 대해 설명해줘"))

            Viscosity  Velocity  PrintingSpeed  PatternSize
Unnamed: 0                                                 
1                   8         5            150           19
5                   5         7             50          224
17                  8         5            125           29
19                  8         7             25           88
24                  5         6            175           61
velocity가 가장 큰 데이터:
Viscosity          8
Velocity           9
PrintingSpeed    250
PatternSize       14
Name: 125, dtype: int64



In [22]:
# REPL을 활용하는 체인
print(repl_execute_chain.invoke("Velocity가 가장 큰 데이터에 대해 설명해줘"))

Velocity가 가장 큰 데이터는 다음과 같습니다.
Viscosity: 8
PrintingSpeed: 250
PatternSize: 14



반환한 데이터도 일종의 문서라고 볼 수 있으므로, RAG 체인을 구성하는 것과 비슷하게 구성하여 챗봇을 구성할 수 있을 것입니다.

In [23]:
def init_chain():
    messages_with_contexts = [
        ("system", "사용자가 입력하는 정보를 바탕으로 질문에 답하세요."),
        ("human", "정보: {context}.\n{question}."),
    ]

    prompt_with_context = ChatPromptTemplate.from_messages(messages_with_contexts)

    # 체인 구성
    qa_chain = (
        {"context": code_execute_chain, "question": RunnablePassthrough()}
        | prompt_with_context
        | llm
        | StrOutputParser()
    )
    
    return qa_chain

In [24]:
code_qa_chain = init_chain()

In [25]:
print(code_qa_chain.invoke("Velocity가 가장 큰 데이터를 찾아줘"))

Velocity의最大값은 9입니다.


## 3. 데이터 그래프 작성 체인 구성
- 데이터를 기반으로 하는 그래프를 그리고 출력하는 체인을 구성합니다.

2챕터와 유사한 방식으로 사용자의 요청에 따라 그래프를 그리는 코드를 생성하고, 그 그래프를 실행하여 그래프를 사용자에게 보여주는 체인을 구성해 봅시다.

In [ ]:
graph_system_message = (
    "당신은 주어진 데이터를 분석하는 데이터 분석가입니다.\n" # 역할(페르소나) 부여
    f"{df_name} DataFrame으로 주어진 데이터를 활용해 요청에 맞는 그래프를 그리는 파이썬 코드를 작성하세요.\n" # 역할 설명
    f"{df_name} DataFrame에는 다음과 같은 열이 있습니다: {df_columns}\n" # 데이터 정보 제공
    "데이터는 이미 로드되어 있으므로 데이터 로드 코드를 생략해야 합니다." # 추가 지시사항
)

message_with_data_info = [
    ("system", graph_system_message),
    ("human", "{question}"),
]

시스템 프롬프트를 확인해 봅시다.

In [ ]:
print(graph_system_message)

In [ ]:
graph_prompt_with_data_info = ChatPromptTemplate.from_messages(message_with_data_info)

# 체인을 구성합니다.
graph_code_execute_chain = (
    {"question": RunnablePassthrough()}
    | prompt_with_data_info
    | llm
    | StrOutputParser()
    | python_code_parser
    | run_code
)

구성한 체인을 실행해 봅시다.

In [ ]:
print(graph_code_execute_chain.invoke("데이터 분포를 그려줘"))

대부분의 경우, 질문에 맞는 그래프가 나타납니다.

이는 체인에 포함된 `run_code` 함수에서 그래프를 그리는 코드 (대부분 `plt.plot()`)를 실행하면서 그래프가 그대로 출력되는 것입니다.

또한, 'plot.png 파일로 저장해줘' 같은 표현을 프롬프트에 추가할 경우, 그린 그래프를 저장하는 코드까지 실행하면서 질문에 대응하는 그래프를 파일로 저장합니다

In [ ]:
print(graph_code_execute_chain.invoke("데이터 분포를 그리고 plot.png 파일로 저장해줘."))

`Ctrl/Cmd+S`를 눌러 노트북 파일을 저장한 후, 왼쪽 위에 있는 Jupyter 로고를 눌러 초기 화면으로 돌아가면 그래프 파일이 저장되어 있을 것입니다.

저희는 특정 데이터의 컬럼명 정보를 프롬프트에 추가한 챗봇을 통해 데이터를 분석하는 코드를 생성했고, 그 코드를 실행한 결과를 다시 챗봇에 입력해서 사용자의 질문에 답하는 챗봇을 개발했습니다.

그리고 다른 챗봇으로 사용자의 요청에 맞는 그래프를 그리는 챗봇도 개발했습니다.

하지만, 현재까지 구현한 챗봇은 상황에 따라 데이터 탐색, 그래프 plot, 데이터 기반 자연어 답변, 단순 답변을 선택하는 기능이 없습니다.

이 한계는 3장 에서 개선해봅시다.